# Featurization

Compute two molecular representations for every compound in the cleaned KIT bioactivity dataset (`02_data_cleaning.ipynb`): ECFP (Morgan) fingerprints via RDKit, and frozen ChemBERTa embeddings via HuggingFace. Both are cached to `data/processed/` so later notebooks (04+) don't need to recompute them.

See [Design Doc.md](../Design%20Doc.md) §5.2 and [IMPLEMENTATION_PLAN.md](../IMPLEMENTATION_PLAN.md) Phase 3. Featurization functions live in [`src/featurization.py`](../src/featurization.py) rather than inline here, since Phase 4/5 notebooks will need to re-featurize new SMILES (e.g. anchor compounds in the selectivity analysis).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append("../src")
from featurization import compute_chemberta_embeddings, compute_ecfp_fingerprints

PROCESSED_DIR = Path("../data/processed")

df = pd.read_csv(PROCESSED_DIR / "kit_bioactivity_clean.csv")
print("Loaded:", df.shape)
df.head()

Loaded: (5565, 10)


,molecule_chembl_id,kit_variant,canonical_smiles,p_value,censored,censored_direction,n_measurements,p_value_std,n_documents,standard_types
0,CHEMBL10,D816V,C[S+]([O-])c1ccc(-c2nc(-c3ccc(F)cc3)c(-c3ccncc...,5.000000,True,>,3.0,NaN,3.0,Kd
1,CHEMBL10,WT,C[S+]([O-])c1ccc(-c2nc(-c3ccc(F)cc3)c(-c3ccncc...,5.000000,True,>,13.0,NaN,4.0,"Kd,Ki"
2,CHEMBL101253,D816V,Clc1ccc(Nc2nnc(Cc3ccncc3)c3ccccc23)cc1,5.000000,True,>,3.0,NaN,3.0,Kd
3,CHEMBL101253,WT,Clc1ccc(Nc2nnc(Cc3ccncc3)c3ccccc23)cc1,6.677781,False,NaN,17.0,1.308074,7.0,"IC50,Kd,Ki"
4,CHEMBL101683,WT,O=C(Nc1ccc(Cl)cc1)c1ccccc1NCc1ccncc1,6.619789,False,NaN,1.0,NaN,1.0,IC50


## 1. Deduplicate SMILES before featurizing

`kit_bioactivity_clean.csv` has one row per `(molecule_chembl_id, kit_variant)` pair (Phase 2), so a compound with both WT and D816V records appears twice with an identical `canonical_smiles`. Featurizing is a function of structure alone, not of variant, so computing it once per unique SMILES and mapping back to the full row order avoids redundant computation (RDKit is cheap, but ChemBERTa forward passes are not) without changing the final row-for-row alignment.

In [2]:
unique_smiles = df["canonical_smiles"].drop_duplicates().tolist()
print(f"{len(unique_smiles)} unique SMILES / {len(df)} total rows")

4640 unique SMILES / 5565 total rows


## 2. Baseline: ECFP (Morgan) fingerprints

Design Doc §5.2: "fast, interpretable-ish, standard cheminformatics baseline." Using radius 2, 2048 bits (the standard ECFP4-equivalent default) via `rdFingerprintGenerator` (RDKit's current, non-deprecated Morgan API).

In [3]:
ecfp_unique = compute_ecfp_fingerprints(unique_smiles, radius=2, n_bits=2048)
print("ECFP shape (unique compounds):", ecfp_unique.shape)
print("Mean bits set per fingerprint:", ecfp_unique.sum(axis=1).mean().round(1))

ECFP shape (unique compounds): (4640, 2048)
Mean bits set per fingerprint: 58.7


## 3. Pretrained chemical language model: ChemBERTa embeddings

Design Doc §5.2/§5.3: ChemBERTa is used **frozen** — embeddings only, no fine-tuning here. Model: `seyonec/ChemBERTa-zinc-base-v1` (RoBERTa architecture pretrained on ~100M ZINC15 SMILES; the standard, most widely-used ChemBERTa checkpoint on HuggingFace). Each compound's embedding is the mean-pooled last hidden state over non-padding tokens (768-dim).

In [4]:
chemberta_unique = compute_chemberta_embeddings(unique_smiles, batch_size=32)
print("ChemBERTa shape (unique compounds):", chemberta_unique.shape)

/Users/leabrody-heine2/anaconda3/envs/kit-inhibitor-binding-prediction/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 18507.98it/s]


[transformers] RobertaModel LOAD REPORT from: seyonec/ChemBERTa-zinc-base-v1
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ChemBERTa shape (unique compounds): (4640, 768)


## 4. Map unique-compound features back to full row order

Both feature matrices above are indexed by `unique_smiles`, not by the original dataframe's row order. Build a SMILES → row-index lookup and expand each matrix to `(len(df), n_features)`, so row *i* of every cached array corresponds to row *i* of `kit_bioactivity_clean.csv` — required for Phase 4 training to line up features with `p_value` labels.

In [5]:
smiles_to_idx = {s: i for i, s in enumerate(unique_smiles)}
row_lookup = df["canonical_smiles"].map(smiles_to_idx).to_numpy()

ecfp_full = ecfp_unique[row_lookup]
chemberta_full = chemberta_unique[row_lookup]

print("ECFP full shape:", ecfp_full.shape)
print("ChemBERTa full shape:", chemberta_full.shape)

assert ecfp_full.shape[0] == len(df)
assert chemberta_full.shape[0] == len(df)

ECFP full shape: (5565, 2048)
ChemBERTa full shape: (5565, 768)


### Sanity-check alignment

Recompute the fingerprint for a few individual rows directly (not via the unique-SMILES lookup path) and confirm they match the corresponding row of `ecfp_full` exactly. This guards against an off-by-one or misordering bug in the lookup/expansion step above.

In [6]:
check_rows = df.sample(5, random_state=0).index
for i in check_rows:
    direct = compute_ecfp_fingerprints([df.loc[i, "canonical_smiles"]], radius=2, n_bits=2048)[0]
    assert np.array_equal(direct, ecfp_full[i]), f"Mismatch at row {i}"
print(f"Verified {len(check_rows)} rows: direct recomputation matches ecfp_full via lookup/expansion.")

Verified 5 rows: direct recomputation matches ecfp_full via lookup/expansion.


## 5. Cache to `data/processed/`

Saved as `.npy` (row order matches `kit_bioactivity_clean.csv`, gitignored as reproducible like the other processed artifacts — regenerate by re-running this notebook).

In [7]:
np.save(PROCESSED_DIR / "ecfp_fingerprints.npy", ecfp_full)
np.save(PROCESSED_DIR / "chemberta_embeddings.npy", chemberta_full)

print("Saved:")
print(f"  ecfp_fingerprints.npy      {ecfp_full.shape}  {ecfp_full.dtype}")
print(f"  chemberta_embeddings.npy  {chemberta_full.shape}  {chemberta_full.dtype}")

Saved:
  ecfp_fingerprints.npy      (5565, 2048)  uint8
  chemberta_embeddings.npy  (5565, 768)  float32


## Summary

- 4,640 unique compounds featurized, expanded to 5,565 rows to match `kit_bioactivity_clean.csv` (one row per `(molecule_chembl_id, kit_variant)`).
- ECFP (Morgan, radius 2, 2048 bits) and ChemBERTa (`seyonec/ChemBERTa-zinc-base-v1`, frozen, mean-pooled, 768-dim) both cached to `data/processed/`.
- Row-for-row alignment with the cleaned dataset verified: shape check against `len(df)` plus a direct-recomputation spot-check on 5 random rows.
- Next: Phase 4 (`04_model_training.ipynb`) — scaffold split, XGBoost-on-ECFP vs. MLP-on-ChemBERTa comparison.